In [16]:
import pandas as pd
import glob
import os
from datetime import datetime

# Search for CSV file starting with 'teal_iq'
csv_files = glob.glob("teal_iq*.csv")
if csv_files:
    csv_file_path = csv_files[0]  # Use the first match
    df = pd.read_csv(csv_file_path, low_memory=False)
else:
    raise FileNotFoundError("No CSV file matching 'teal_iq*' was found.")

# Filter based on 'cert_' keyword
df_cert = df[df['enriched_field'].str.contains('cert_', case=False, na=False)]
required_headers = ['cert_category', 'cert_domain', 'cert_expiration_date', 'cert_id', 'cert_number', 'cert_subcategory', 'cert_type', 'certifying_body']

# List to store all rows
all_rows = []

# Loop through each `merged_value_#` column
merged_columns = [col for col in df_cert.columns if col.startswith('merged_value_')]

for col in merged_columns:
    temp_df = df_cert[['tealbook_id', 'internal_supplier_id_or_vendor_number', 'enriched_field', col]].copy()
    pivoted_df = temp_df.pivot_table(index=('internal_supplier_id_or_vendor_number', 'tealbook_id'), columns='enriched_field', values=col, aggfunc='first')
    pivoted_df = pivoted_df.reindex(columns=required_headers)
    all_rows.append(pivoted_df.reset_index())

# Concatenate all DataFrames together
qa_less_trust = pd.concat(all_rows, ignore_index=True)

# Search for Excel file containing 'VM Analysis'
excel_files = glob.glob("*VM Analysis*.xlsx")
if excel_files:
    excel_file_path = excel_files[0]  # Use the first match
    vm_analysis = pd.read_excel(excel_file_path, sheet_name='Spend Summary')
else:
    raise FileNotFoundError("No Excel file matching '*VM Analysis*' was found.")

qa_less_trust = qa_less_trust[qa_less_trust['cert_category'] == 'diversity']
df_trust = df_cert.loc[df_cert['enriched_field'] == 'cert_id']

columns_to_keep = ['internal_supplier_id_or_vendor_number', 'tealbook_id'] + \
                  [col for col in df_cert.columns if col.startswith('merged_value_') or col.startswith('tealbook_trust_score_')]

final_filtered_df = df_trust[columns_to_keep]
melted_df = pd.melt(final_filtered_df, id_vars=['internal_supplier_id_or_vendor_number', 'tealbook_id'], 
                    value_vars=[col for col in final_filtered_df.columns if col.startswith('tealbook_trust_score_') or col.startswith('merged_value_')],
                    var_name='variable', 
                    value_name='value')

melted_df['type'] = melted_df['variable'].apply(lambda x: 'trust_score' if 'tealbook_trust_score' in x else 'merged_value')
melted_df['index'] = melted_df['variable'].str.extract('(\d+)')

final_combined_df = melted_df.pivot_table(index=['internal_supplier_id_or_vendor_number', 'tealbook_id', 'index'], 
                                           columns='type', 
                                           values='value', 
                                           aggfunc='first').reset_index()

final_combined_df.drop(columns='index', inplace=True)
merged_df = pd.merge(
    qa_less_trust,
    final_combined_df,
    how='left',
    left_on=['internal_supplier_id_or_vendor_number', 'cert_id'],
    right_on=['internal_supplier_id_or_vendor_number', 'merged_value'],
    suffixes=('', '_final')
)

merged_df.drop(columns='merged_value', inplace=True)

company_name_df = df[df['enriched_field'] == 'company_name'][['internal_supplier_id_or_vendor_number', 'merged_value_1']]
company_name_df = company_name_df.rename(columns={'merged_value_1': 'company_name'})

company_address_df = df[df['enriched_field'] == 'complete_address'][['internal_supplier_id_or_vendor_number', 'merged_value_1']]
company_address_df = company_address_df.rename(columns={'merged_value_1': 'company_address'})

merged_df = pd.merge(merged_df, company_name_df, on='internal_supplier_id_or_vendor_number', how='left')
merged_df = pd.merge(merged_df, company_address_df, on='internal_supplier_id_or_vendor_number', how='left')

cols = merged_df.columns.tolist()
cols.insert(2, cols.pop(cols.index('company_name')))
cols.insert(3, cols.pop(cols.index('company_address')))
merged_df = merged_df[cols]

# Ensure 'internal supplier id' remains as string
vm_analysis['internal supplier id'] = vm_analysis['internal supplier id'].astype(str)

# Similarly, ensure 'internal_supplier_id_or_vendor_number' is also treated as string
merged_df['internal_supplier_id_or_vendor_number'] = merged_df['internal_supplier_id_or_vendor_number'].astype(str)

# Perform the merge operation
merged_df1 = pd.merge(merged_df, vm_analysis[['internal supplier id', 'aggregated spend']],
                      left_on='internal_supplier_id_or_vendor_number',
                      right_on='internal supplier id', how='left')

merged_df1 = merged_df1[merged_df1['aggregated spend'].notna()]

exp_date_df = merged_df.copy()
exp_date_df['cert_expiration_date'] = pd.to_datetime(exp_date_df['cert_expiration_date'], errors='coerce')
today = pd.Timestamp(datetime.now())

exp_date_df['days_diff'] = (exp_date_df['cert_expiration_date'] - today).dt.days

next_3_months = today + pd.DateOffset(months=3)
following_3_months = today + pd.DateOffset(months=6)

def categorize_date(days_diff):
    if pd.isna(days_diff):
        return "Invalid date"
    elif 0 > days_diff >= -30:
        return "expired 0-30 days"
    elif -31 >= days_diff >= -90:
        return "expired 31-90 days"
    elif -91 >= days_diff >= -180:
        return "expired 91-180 days"
    elif -181 >= days_diff >= -365:
        return "expired 181-365 days"
    elif -366 >= days_diff:
        return "expired 365+ days"
    elif 0 <= days_diff <= 90:
        return "Next 3 months"
    else:
        return "Post following 3 months"

exp_date_df['category'] = exp_date_df['days_diff'].apply(categorize_date)
category_counts = exp_date_df['category'].value_counts()
total_count = len(exp_date_df)
category_counts = pd.concat([category_counts, pd.Series({'Total': total_count})])
percentage_counts = (category_counts / total_count * 100).round(2)
percentage_counts.name = 'Percentage'

category_summary = pd.DataFrame({
    'Count': category_counts,
    'Percentage': percentage_counts
})

new_order = [
    'Invalid date', 'Next 3 months', 'Post following 3 months',
    'expired 0-30 days', 'expired 31-90 days', 'expired 91-180 days',
    'expired 181-365 days', 'expired 365+ days', 'Total'
]

category_summary = category_summary.reindex(new_order)

df
unique_certs = merged_df['cert_subcategory'].unique()

sorted_df = merged_df1.sort_values(by=['internal_supplier_id_or_vendor_number', 'cert_subcategory'])
unique_cert_df = sorted_df.drop_duplicates(subset=['internal_supplier_id_or_vendor_number', 'cert_subcategory'], keep='first')

spend_by_cert = unique_cert_df.groupby('cert_subcategory')['aggregated spend'].sum()
total_spend = spend_by_cert.sum()
spend_percentage = (spend_by_cert / total_spend) * 100

unique_items_list = ['sbe', 'sdb', 'wbe', 'dbe', 'vbe', 'sdvbe', 'mbe', 'hub', 'pwd', 'lgbtbe', 'cab', 'hbcu']

spend_filtered = spend_by_cert.reindex(unique_items_list, fill_value=0)
percentage_filtered = spend_percentage.reindex(unique_items_list, fill_value=0)

counts_filtered_unique_ids = merged_df1.groupby('cert_subcategory')['internal_supplier_id_or_vendor_number'].nunique().reindex(unique_items_list, fill_value=0)

result_df = pd.DataFrame({
    'Cert Counts': counts_filtered_unique_ids,
    'Spend': spend_filtered,
    'Percentage of Total Spend (%)': percentage_filtered
})

ordered_list = ['sbe', 'sdb', 'wbe', 'dbe', 'vbe', 'sdvbe', 'mbe', 'hub', 'pwd', 'lgbtbe', 'cab', 'hbcu']
priority_mapping = {cert: index for index, cert in enumerate(ordered_list)}

merged_df1['priority'] = merged_df1['cert_subcategory'].map(priority_mapping)
sorted_df = merged_df1.sort_values(by=['internal_supplier_id_or_vendor_number', 'priority'])
unique_cert_df = sorted_df.drop_duplicates(subset='internal_supplier_id_or_vendor_number', keep='first')
unique_cert_df = unique_cert_df.drop(columns='priority')

spend_by_cert = unique_cert_df.groupby('cert_subcategory')['aggregated spend'].sum()
counts_by_cert = unique_cert_df.groupby('cert_subcategory')['internal_supplier_id_or_vendor_number'].nunique()
total_spend = spend_by_cert.sum()
spend_percentage = (spend_by_cert / total_spend) * 100

spend_filtered = spend_by_cert.reindex(unique_items_list, fill_value=0)
counts_filtered = counts_by_cert.reindex(unique_items_list, fill_value=0)
percentage_filtered = spend_percentage.reindex(unique_items_list, fill_value=0)

final_result_df = pd.DataFrame({
    'Cert Counts': counts_filtered,
    'Total Spend': spend_filtered,
    'Percentage of Total Spend (%)': percentage_filtered
})

supplier_ids = df['internal_supplier_id_or_vendor_number'].unique()

def get_consolidated_diversity(supplier_id, df):
    filtered_rows = df[(df['internal_supplier_id_or_vendor_number'] == supplier_id) &
                       (df['enriched_field'] == 'cert_subcategory')]
    if not filtered_rows.empty:
        merged_values = filtered_rows[[
            'merged_value_1', 'merged_value_2', 'merged_value_3', 'merged_value_4', 
            'merged_value_5', 'merged_value_6', 'merged_value_7', 'merged_value_8', 
            'merged_value_9', 'merged_value_10'
        ]].values.flatten()
        non_empty_values = [str(val) for val in merged_values if pd.notnull(val) and val != '']
        consolidated = '|'.join(non_empty_values)
        unique_values = '|'.join(sorted(set(consolidated.split('|'))))
        return unique_values
    return ''

def get_certification_trust_value(supplier_id, df, cert_name):
    supplier_data = df[df['internal_supplier_id_or_vendor_number'] == supplier_id]
    if not supplier_data.empty:
        merged_values = supplier_data[[
            'merged_value_1', 'merged_value_2', 'merged_value_3', 'merged_value_4', 
            'merged_value_5', 'merged_value_6', 'merged_value_7', 'merged_value_8', 
            'merged_value_9', 'merged_value_10'
        ]].values.flatten()
        trust_scores = supplier_data[[
            'tealbook_trust_score_1', 'tealbook_trust_score_2', 'tealbook_trust_score_3',
            'tealbook_trust_score_4', 'tealbook_trust_score_5', 'tealbook_trust_score_6',
            'tealbook_trust_score_7', 'tealbook_trust_score_8', 'tealbook_trust_score_9', 
            'tealbook_trust_score_10'
        ]].values.flatten()

        for i, value in enumerate(merged_values):
            if pd.notnull(value) and str(value).strip().lower() == cert_name.lower():
                return trust_scores[i]  # Return the corresponding trust score
    return 'n/a'

diversity_report = []
for supplier_id in supplier_ids:
    company_name = df.loc[
        (df['internal_supplier_id_or_vendor_number'] == supplier_id) & 
        (df['enriched_field'] == 'company_name'), 
        'merged_value_1'
    ].values

    supplier_name = str(company_name[0]) if len(company_name) > 0 else None

    company_address = df.loc[
        (df['internal_supplier_id_or_vendor_number'] == supplier_id) & 
        (df['enriched_field'] == 'complete_address'), 
        'merged_value_1'
    ].values

    supplier_address = str(company_address[0]) if len(company_address) > 0 else None

    consolidated_diversity = get_consolidated_diversity(supplier_id, df)
    
    sbe_trust_value = get_certification_trust_value(supplier_id, df, 'sbe')
    sdb_trust_value = get_certification_trust_value(supplier_id, df, 'sdb')
    mbe_trust_value = get_certification_trust_value(supplier_id, df, 'mbe')
    wbe_trust_value = get_certification_trust_value(supplier_id, df, 'wbe')
    vbe_trust_value = get_certification_trust_value(supplier_id, df, 'vbe')
    sdvbe_trust_value = get_certification_trust_value(supplier_id, df, 'sdvbe')
    hub_trust_value = get_certification_trust_value(supplier_id, df, 'hub')
    dbe_trust_value = get_certification_trust_value(supplier_id, df, 'dbe')
    cab_trust_value = get_certification_trust_value(supplier_id, df, 'cab')
    pwd_trust_value = get_certification_trust_value(supplier_id, df, 'pwd')
    lgbtbe_trust_value = get_certification_trust_value(supplier_id, df, 'lgbtbe')
    dvbe_trust_value = get_certification_trust_value(supplier_id, df, 'dvbe')
    hbcu_trust_value = get_certification_trust_value(supplier_id, df, 'hbcu')

    diversity_report.append({
        'Internal Supplier ID': supplier_id,
        'Supplier Name': supplier_name,
        'Supplier Address': supplier_address,
        'Consolidated Diversity': consolidated_diversity,
        'SBE (Trust/n/a)': sbe_trust_value,
        'SDB (Trust/n/a)': sdb_trust_value,
        'MBE (Trust/n/a)': mbe_trust_value,
        'WBE (Trust/n/a)': wbe_trust_value,
        'VBE (Trust/n/a)': vbe_trust_value,
        'SDVBE (Trust/n/a)': sdvbe_trust_value,
        'HUB (Trust/n/a)': hub_trust_value,
        'DBE (Trust/n/a)': dbe_trust_value,
        'CAB (Trust/n/a)': cab_trust_value,
        'PWD (Trust/n/a)': pwd_trust_value,
        'LGBTBE (Trust/n/a)': lgbtbe_trust_value,
        'DVBE (Trust/n/a)': dvbe_trust_value
    })

filename = f"Step 7_SDP Diversity Snap - {datetime.now().strftime('%d-%m-%Y')}.xlsx"
diversity_df = pd.DataFrame(diversity_report)

# Create a Pandas Excel writer using XlsxWriter as the engine
with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
    workbook = writer.book
    teal_header_format = workbook.add_format({'bold': True, 'bg_color': '#008080', 'font_color': 'white', 'border': 1})
    merged_df1.to_excel(writer, sheet_name='data', index=False)
    worksheet2 = writer.sheets['data']
    worksheet2.set_column('A:Z', 50)
    
    for col_num, value in enumerate(merged_df1.columns.values):
        worksheet2.write(0, col_num, value, teal_header_format)

    result_df.to_excel(writer, sheet_name='Certification Spend', startrow=1, startcol=0, index=True)
    final_result_df.to_excel(writer, sheet_name='Certification Spend', startrow=len(result_df) + 5, startcol=0, index=True)

    workbook = writer.book
    teal_header_format = workbook.add_format({'bold': True, 'bg_color': '#008080', 'font_color': 'white', 'border': 1})
    title_format = workbook.add_format({
        'align': 'center', 'valign': 'vcenter', 'bold': True, 'font_size': 14
    })
    
    worksheet3 = writer.sheets['Certification Spend']
    title = "Qualified cert list"
    title1 = "Priority cert list"
    worksheet3.merge_range('A1:D1', title, title_format)
    worksheet3.merge_range(f'A{len(result_df) + 5}:D{len(result_df) + 5}', title1, title_format)
    worksheet3.set_column('A:A', 20)
    worksheet3.set_column('B:C', 15)
    worksheet3.set_column('D:D', 25)
    
    light_border_format = workbook.add_format({
    'border': 1,  
    'border_color': '#D3D3D3'
    })
    percentage_format = workbook.add_format({
    'num_format': '0.00%',
    'border': 1,  
    'border_color': '#D3D3D3'
    })

    worksheet3.write(1, 0, result_df.index.name or '', teal_header_format)  
    for col_num, value in enumerate(result_df.columns.values):
        worksheet3.write(1, col_num + 1, value, teal_header_format)

    for row_num in range(2, len(result_df) + 2):  
        worksheet3.write(row_num, 0, result_df.index[row_num - 2], light_border_format)
        for col_num, value in enumerate(result_df.iloc[row_num - 2]):  
            if col_num == 2:  
                worksheet3.write(row_num, col_num + 1, value / 100, percentage_format)
            else:
                worksheet3.write(row_num, col_num + 1, value, light_border_format)

    start_row = len(result_df) + 5
    worksheet3.write(start_row, 0, final_result_df.index.name or '', teal_header_format)
    for col_num, value in enumerate(final_result_df.columns.values):
        worksheet3.write(start_row, col_num + 1, value, teal_header_format)

    for row_num in range(start_row + 1, start_row + len(final_result_df) + 1):  
        worksheet3.write(row_num, 0, final_result_df.index[row_num - (start_row + 1)], light_border_format)
        for col_num, value in enumerate(final_result_df.iloc[row_num - (start_row + 1)]):
            if col_num == 2:  
                worksheet3.write(row_num, col_num + 1, value / 100, percentage_format)
            else:
                worksheet3.write(row_num, col_num + 1, value, light_border_format)

    workbook = writer.book
    teal_header_format = workbook.add_format({'bold': True, 'bg_color': '#008080', 'font_color': 'white', 'border': 1})
    diversity_df.to_excel(writer, sheet_name='Diversity Report', index=False)
    worksheet1 = writer.sheets['Diversity Report']
    worksheet1.set_column('B:B', 20)
    worksheet1.set_column('C:C', 50)
    worksheet1.set_column('D:D', 20)
    worksheet1.set_column('E:P', 15)
    
    for col_num, value in enumerate(diversity_df.columns.values):
        worksheet1.write(0, col_num, value, teal_header_format)

# No need to call writer.close() here
print(f"Excel file '{filename}' has been created with the NAICS List tab, sorted NAICS column, and datestamp.")

# If you intended to enumerate through the columns, use this:
for index, column in enumerate(result_df.columns.values, start=1):
    print(index, column)


Excel file '17-10-2024 SDP QA Snap.xlsx' has been created with the NAICS List tab, sorted NAICS column, and datestamp.
1 Cert Counts
2 Spend
3 Percentage of Total Spend (%)
